In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Load csv files and combine into one dataframe
dfs = []
for year in range(2020, 2026):
    df = pd.read_csv(f'data/{year}.csv', encoding='latin-1')
    dfs.append(df)

tube_df = pd.concat(dfs, ignore_index=True)

In [4]:
# Reformat time to date time object
tube_df['Start Date Time'] = pd.to_datetime(tube_df['Start Date Time'], dayfirst=True)
tube_df['month'] = tube_df['Start Date Time'].dt.to_period('M').dt.to_timestamp()

# Group by for charting
time_df = tube_df.groupby(['month', 'Service Status']).size().reset_index(name='count')
time_df['month'] = pd.to_datetime(time_df['month'])

# Define status order for stacking (worst at top)
status_order = ['Suspended', 'Severe Delays', 'Part Suspended', 'Part Closure', 'Planned Closure', 'Minor Delays']

# Filter to disruption statuses, after covid, and before February 2025 to account for incomplete month
time_df = time_df[time_df['Service Status'].isin(status_order)]
time_df = time_df[time_df['month'].dt.year >= 2021]
time_df = time_df[time_df['month'] < '2025-02-01']
time_df

,month,Service Status,count
69,2021-01-01,Minor Delays,429
70,2021-01-01,Part Closure,132
71,2021-01-01,Part Suspended,309
72,2021-01-01,Planned Closure,55
73,2021-01-01,Severe Delays,307
...,...,...,...
367,2025-01-01,Part Closure,226
368,2025-01-01,Part Suspended,268
369,2025-01-01,Planned Closure,24
370,2025-01-01,Severe Delays,476


In [5]:
# Plot disruptions over time with status breakdown
disrupts_line = alt.Chart(time_df).mark_area(opacity=0.9).encode(
    x=alt.X('month:T', 
            title=None, 
            axis=alt.Axis(
                format='%Y', 
                tickCount='year',
                grid=False)),
    y=alt.Y('count:Q', 
            stack=True, 
            title='Number of disruptions',
            axis=alt.Axis(grid=False)),
    color=alt.Color('Service Status:N',
                    scale=alt.Scale(domain=status_order),
                    sort=status_order),
                    order=alt.Order('Service Status:N', sort='descending')
)

disrupts_line

alt.Chart(...)

In [6]:
time_df.groupby('month')['count'].sum().sort_values(ascending=False).head(10)

month
2021-07-01    2017
2024-08-01    2016
2022-12-01    1967
2024-07-01    1957
2024-06-01    1894
2024-03-01    1874
2024-11-01    1853
2023-09-01    1822
2023-06-01    1813
2022-04-01    1792
Name: count, dtype: int64

In [7]:
time_df.groupby([time_df['month'].dt.year, 'Service Status'])['count'].sum().unstack()

Service Status,Minor Delays,Part Closure,Part Suspended,Planned Closure,Severe Delays,Suspended
month,,,,,,
2021,6302,1768,3314,703,4676,269
2022,6878,1428,4161,372,4951,471
2023,7593,1066,3825,667,5487,198
2024,8674,1499,4104,550,6098,181
2025,613,226,268,24,476,12


In [8]:
yearly = time_df.groupby([time_df['month'].dt.year, 'Service Status'])['count'].sum().unstack()
yearly_pct_change = yearly.pct_change() * 100
yearly_pct_change.round(2)

Service Status,Minor Delays,Part Closure,Part Suspended,Planned Closure,Severe Delays,Suspended
month,,,,,,
2021,NaN,NaN,NaN,NaN,NaN,NaN
2022,9.14,-19.23,25.56,-47.08,5.88,75.09
2023,10.40,-25.35,-8.07,79.30,10.83,-57.96
2024,14.24,40.62,7.29,-17.54,11.14,-8.59
2025,-92.93,-84.92,-93.47,-95.64,-92.19,-93.37


Disruptions on the London Underground have increased steadily since 2021, with 2024 the worst year on record. Five of the ten most disrupted months fell in 2024, and severe delays have risen every year since 2021.

In [9]:
# Save charts
styles.save(disrupts_line, 'visualisation', 'disruptions_line_chart', width=450, height=360)